In [ ]:
from mpl_toolkits import mplot3d
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
from torchinfo import summary

In [ ]:
# CNN Model
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.main = nn.Sequential(
            torch.nn.Conv2d(in_channels=1, out_channels=32, kernel_size=(3,3), padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=(2,2)),
            torch.nn.Conv2d(in_channels=32, out_channels=64, kernel_size=(3,3), padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=(2,2)),
            torch.nn.Flatten(),
            torch.nn.Linear(64*16*16, 49),
        )
    def forward(self, x):
        out = self.main(x)
        return out

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
model=CNN()
model=model.to(device)
summary(model, (1, 1, 64, 64))  # Example input size: batch_size=1, channels=1, height=64, width=64

In [ ]:
# KMNIST dataset loading
transform = torchvision.transforms.Compose([
        torchvision.transforms.Resize((64, 64)),
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize((0.1307,), (0.3081,))])
    

batch_size = 64
trainset = torchvision.datasets.KMNIST(
    root='./archive', train=True, download=True,
    transform=transform
)

trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=batch_size, shuffle=True, num_workers=2
)

testset = torchvision.datasets.KMNIST(
    root='./archive', train=False, download=True,
    transform=transform
)

testloader = torch.utils.data.DataLoader(
    testset, batch_size=batch_size, shuffle=False, num_workers=2
)

classes = trainset.classes

In [ ]:
# Custom dataset for K49 from .npz files
from torch.utils.data import Dataset
from PIL import Image
import numpy as np

class K49Dataset(Dataset):
    def __init__(self, imgs_file, labels_file, transform=None):
        """
            imgs_file: Path to the .npz file with images
            labels_file: Path to the .npz file with labels
            transform: Optional transform to be applied on a sample
        """
        imgs_data = np.load(imgs_file)
        labels_data = np.load(labels_file)
        
        self.images = imgs_data['arr_0'] 
        self.labels = labels_data['arr_0']
        self.transform = transform
        
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        image = self.images[idx]
        label = int(self.labels[idx])
        
        image = Image.fromarray(image)
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

In [ ]:
# Create datasets from k-49 files with data augmentation for training
train_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize((64, 64)),
    torchvision.transforms.RandomRotation(degrees=15), 
    torchvision.transforms.RandomAffine(
        degrees=0,
        translate=(0.1, 0.1), 
        scale=(0.9, 1.1) 
    ),
    torchvision.transforms.RandomPerspective(distortion_scale=0.2, p=0.5),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize((0.1307,), (0.3081,))
])

test_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize((64, 64)),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize((0.1307,), (0.3081,))
])

trainset = K49Dataset(
    imgs_file='archive/k49-train-imgs.npz',
    labels_file='archive/k49-train-labels.npz',
    transform=train_transform 
)

testset = K49Dataset(
    imgs_file='archive/k49-test-imgs.npz',
    labels_file='archive/k49-test-labels.npz',
    transform=test_transform
)

batch_size = 64

trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=batch_size, shuffle=True, num_workers=2
)

testloader = torch.utils.data.DataLoader(
    testset, batch_size=batch_size, shuffle=False, num_workers=2
)

classmap = pd.read_csv('archive/k49_classmap.csv')
print(classmap.head())
classes = classmap.iloc[:, 2].tolist()

In [ ]:
def imshow(img):
    img = img / 2 + 0.5     # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

dataiter = iter(trainloader)
images, labels = next(dataiter)

imshow(torchvision.utils.make_grid(images))
print(' '.join(f'{classes[labels[j]]:5s}' for j in range(batch_size)))

In [ ]:
# Define a loss function and optimizer
import torch.optim as optim
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

In [ ]:
# Train the network
for epoch in range(50): 
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % 2000 == 1999:
            print(f'[Epoch {epoch + 1}, Mini-batch {i + 1}] loss: {running_loss / 2000:.3f}')
            running_loss = 0.0

In [ ]:
# Set path to save and load the model
PATH = './k49_cnn_50_epochs_augmented.pth'

In [ ]:
# Save the trained model
torch.save(model.state_dict(), PATH)

In [ ]:
# Load the saved model
model = CNN()
model.load_state_dict(torch.load(PATH))
model.to(device)

In [ ]:
# Test the network on the test data
dataiter = iter(testloader)
images, labels = next(dataiter)

imshow(torchvision.utils.make_grid(images))
print('GroundTruth: ', ' '.join(f'{classes[labels[j].item()]}' for j in range(8)))
for j in range(2,9):
    print('             ', ' '.join(f'{classes[labels[j].item()]}' for j in range((j-1)*8, j*8)))


In [ ]:
# Test the network on the above data
outputs = model(images.to(device))
_, predicted = torch.max(outputs, 1)
print('Predicted:   ', ' '.join(f'{classes[predicted[j]]:5s}' for j in range(8)))
for j in range(2,9):
    print('             ', ' '.join(f'{classes[predicted[j]]:5s}' for j in range((j-1)*8, j*8)))

In [ ]:
# Test the data on the whole test dataset
correct = 0
total = 0

with torch.no_grad():
    for data in testloader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        outputs = model(images.to(device))
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the {total} test images: {100 * correct // total} %')

In [ ]:
# Show results per class
correct_pred = {classname: 0 for classname in classes}
total_pred = {classname: 0 for classname in classes}

with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = model(images.to(device))
        _, predictions = torch.max(outputs, 1)
        for label, prediction in zip(labels, predictions):
            if label == prediction:
                correct_pred[classes[label]] += 1
            total_pred[classes[label]] += 1

for classname, correct_count in correct_pred.items():
    accuracy = 100 * float(correct_count) / total_pred[classname]
    print(f'Accuracy for class: {classname:5s} is {accuracy:.1f} %')